## ⚠️ Note

This notebook is a cleaned and modularized version of experiments
conducted in Google Colab. Paths and configurations are simplified
for clarity and documentation purposes.

Some training steps are computationally expensive and may not be
reproducible without the original dataset and GPU resources.


## Comparison: Base Model vs LoRA (Megamendung)

This section compares image generation results between:
- **Base Stable Diffusion model (without LoRA)**
- **Base model enhanced with Megamendung LoRA**

To ensure a fair comparison:
- The **same prompt** is used for both models
- The **same random seed** is fixed to control noise initialization

Any visual differences observed are therefore attributed to the effect of **LoRA fine-tuning**.


In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "GPU is required to run this notebook."

OUTPUT_DIR = "../../results/comparison_lora_no_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Environment ready")
print("📁 Output directory:", OUTPUT_DIR)


In [ ]:
# =========================
# LOAD BASE MODEL
# =========================
BASE_MODEL = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16
).to("cuda")

pipe.enable_attention_slicing()

print("✅ Base model loaded (no LoRA)")

In [ ]:
# =========================
# LOAD MODEL + LoRA
# =========================
BASE_MODEL = "runwayml/stable-diffusion-v1-5"
LORA_DIR = ".../megamendung"
LORA_FILE = "(...).safetensors" # Ex. pytorch_lora_weights.safetensors

pipe_lora = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16
).to("cuda")

pipe_lora.load_lora_weights(
    LORA_DIR,
    weight_name=LORA_FILE
)

pipe_lora.enable_attention_slicing()

print("✅ LoRA loaded correctly (local path)")


In [ ]:
# =========================
# CONFIG
# =========================
PROMPT = (
    "traditional Indonesian batik megamendung pattern, "
    "cloud motif, Cirebon style, textile design, high detail, 4k, red and black"
)

NEG_PROMPT = "blurry, low quality, distorted"
SEED = 39
generator = torch.Generator("cuda").manual_seed(SEED)

# =========================
# GENERATE IMAGES
# =========================
image_base = pipe(
    PROMPT,
    negative_prompt=NEG_PROMPT,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator
).images[0]
image_base.save(f"{OUTPUT_DIR}/compare_baseline_megamendung.png")
generator = torch.Generator("cuda").manual_seed(SEED)

image_lora = pipe_lora(
    PROMPT,
    negative_prompt=NEG_PROMPT,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator
).images[0]
image_lora.save(f"{OUTPUT_DIR}/compare_lora_megamendung.png")


# =========================
# PLOT COMPARISON
# =========================
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(image_base)
plt.title("Without LoRA (Base Model)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(image_lora)
plt.title("With Megamendung LoRA")
plt.axis("off")

plt.suptitle(
    f"Prompt: '{PROMPT}'",
    fontsize=10
)

plt.show()